# 📥 Phase 0: MS Lesion Dataset Downloader (aria2c Optimized)
This notebook downloads all required datasets directly to your Google Drive using **aria2c** for maximum speed and reliability. Run this **once** before starting training.

In [ ]:
import os

# ─── MANUAL OVERRIDE ─────────────────────────────────────────────────────────
ENVIRONMENT = "local"   # Change to "colab" when running on Colab
# ─────────────────────────────────────────────────────────────────────────────

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/brain_dataset"
else:
    SAVE_DIR = "./data/raw"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Environment : {ENVIRONMENT}")
print(f"Data will be saved to: {SAVE_DIR}")

In [ ]:
# ── COLAB MODE: Download from public URLs using aria2c ───────────────────────
if ENVIRONMENT == "colab":
    datasets = {
        "MSLesSeg": {
            "path": os.path.join(SAVE_DIR, "MSLesSeg"),
            "links": [
                ("https://springernature.figshare.com/ndownloader/files/52771814", "MSLesSeg_Part1.zip"),
                ("https://springernature.figshare.com/ndownloader/files/53623946", "MSLesSeg_Part2.zip"),
                ("https://springernature.figshare.com/ndownloader/files/53623967", "MSLesSeg_Part3.zip"),
                ("https://springernature.figshare.com/ndownloader/files/54605591", "MSLesSeg_Part4.zip"),
                ("https://springernature.figshare.com/ndownloader/files/54605609", "MSLesSeg_Part5.zip")
            ]
        },
        "Mendeley": {
            "path": os.path.join(SAVE_DIR, "Mendeley"),
            "links": [
                ("https://data.mendeley.com/public-files/datasets/8bctsm8jz7/files/9356efeb-dcd8-4213-a2d4-8febe9f1a5db/file_downloaded", "Mendeley_Data.zip")
            ]
        },
        "Long-MR-MS": {
            "path": os.path.join(SAVE_DIR, "Long-MR-MS"),
            "links": [
                ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_demographics.csv", "demographics.csv"),
                ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient01-05.zip", "Long_Part1.zip"),
                ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient06-10.zip", "Long_Part2.zip"),
                ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient11-15.zip", "Long_Part3.zip"),
                ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient16-20.zip", "Long_Part4.zip")
            ]
        }
    }

    import requests
    from tqdm.notebook import tqdm

    def download_robustly(url, target_dir, filename):
        target_file = os.path.join(target_dir, filename)
        if os.path.exists(target_file) and os.path.getsize(target_file) > 0:
            print(f"  ✅ {filename} already exists.")
            return
        print(f"  🚀 Downloading: {filename}...")
        session = requests.Session()
        try:
            response = session.get(url, allow_redirects=True, stream=True)
            response.raise_for_status()
            total_size = int(response.headers.get('content-length', 0))
            with open(target_file, "wb") as f:
                with tqdm(total=total_size, unit='B', unit_scale=True, desc=filename, leave=False) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            print(f"  ✨ Saved {filename}")
        except Exception as e:
            print(f"  ❌ Error downloading {filename}: {e}")

    !apt-get install -y aria2

    for name, info in datasets.items():
        print(f"\n📂 Downloading {name}...")
        os.makedirs(info['path'], exist_ok=True)
        for link, filename in info['links']:
            if name == "MSLesSeg":
                download_robustly(link, info['path'], filename)
            else:
                target_file = os.path.join(info['path'], filename)
                if not os.path.exists(target_file) or os.path.getsize(target_file) == 0:
                    print(f"  🚀 aria2c Downloading: {filename}...")
                    !aria2c -x 16 -s 16 -k 1M "{link}" -d "{info['path']}" -o "{filename}"
                else:
                    print(f"  ✅ {filename} already exists.")

    print("\n✅ All datasets downloaded to Drive.")

In [ ]:
# ── LOCAL MODE: Download ZIPs from Google Drive share links using gdown ───────
# For each file: right-click in Drive → "Get link" → "Anyone with the link" → Copy
# Paste the full link (https://drive.google.com/file/d/…/view?usp=sharing)
GDRIVE_SHARE_LINKS = {
    "MSLesSeg": [
        ("https://drive.google.com/file/d/1_IB5uTdwvjJaWP3k-IpY_vqu_Majc9Xf/view", "MSLesSeg_Part1.zip"),
        ("https://drive.google.com/file/d/1C0S6roiSB2_fOKJdyg6soR7IYnCoQnBv/view", "MSLesSeg_Part2.zip"),
        ("https://drive.google.com/file/d/1aYj_P3kpkNYUEqsCxGQsjA81N-UzFmV3/view", "MSLesSeg_Part3.zip"),
        ("https://drive.google.com/file/d/1oUYQGv4dh4Xl21fmfAqDcqVR12c6NraD/view", "MSLesSeg_Part4.zip"),
    ],
    "Mendeley": [
        ("https://drive.google.com/file/d/1GQHSb1jQxb1NQa-piMYzHnTHO_ccHSzw/view", "Mendeley_Data.zip"),
    ],
    "Long-MR-MS": [
        ("https://drive.google.com/file/d/1neKFcntR2G4ljKyVLzxuK3B8ncumS_ag/view", "Long_Part1.zip"),
        ("https://drive.google.com/file/d/1c0myucExxuSdQM2F66Raydk-tsdteoAm/view", "Long_Part2.zip"),
        ("https://drive.google.com/file/d/1cHCnrh3IPA1hoqCDAnYA6PSjEzP4MFm5/view", "Long_Part3.zip"),
        ("https://drive.google.com/file/d/1TGpGAPxB26jxFYtfPD_H3wexZNvDCo4b/view", "Long_Part4.zip"),
    ],
}

if ENVIRONMENT == "local":
    import subprocess, zipfile

    try:
        import gdown
    except ImportError:
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown

    for category, file_list in GDRIVE_SHARE_LINKS.items():
        dest = os.path.join(SAVE_DIR, category)
        os.makedirs(dest, exist_ok=True)
        for url, filename in file_list:
            if "PASTE_ID_HERE" in url:
                print(f"  ⚠️  Skipping {filename} — replace PASTE_ID_HERE with real link")
                continue
            target = os.path.join(dest, filename)
            if not os.path.exists(target) or os.path.getsize(target) == 0:
                print(f"  ⬇️  Downloading {filename}...")
                gdown.download(url, target, quiet=False, fuzzy=True)
            else:
                print(f"  ✅ {filename} already downloaded.")
            if filename.endswith('.zip'):
                flag = target.replace('.zip', '.extracted')
                if not os.path.exists(flag):
                    print(f"  📦 Extracting {filename}...")
                    with zipfile.ZipFile(target, 'r') as z:
                        z.extractall(dest)
                    open(flag, 'w').close()
                else:
                    print(f"  ✅ {filename} already extracted.")

    print("\n✅ Local dataset ready at:", SAVE_DIR)